<a href="https://colab.research.google.com/github/nesbita/ML-Collision-Severity-YOLO-Detection/blob/main/Ariana_Nesbit_Ass2_T23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U ultralytics
from ultralytics import YOLO
import torch
import pandas as pd
print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPU available: True
Device: Tesla T4


In [ ]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

CAR_ONLY_PATH = '/content/drive/MyDrive/kitti3k_car_only'

shutil.copytree(CAR_ONLY_PATH, '/content/kitti3k_car_only')
print('Dataset copied to local storage.')
print(os.listdir('/content/kitti3k_car_only'))

Mounted at /content/drive
Dataset copied to local storage.
['kitti3k_car_only.yaml', 'images', 'labels']


In [ ]:
## 3RD CONFIGURATON TRAINING

car_only_yaml = '/content/kitti3k_car_only/kitti3k_car_only.yaml'

model_car = YOLO('/content/drive/MyDrive/config3_best.pt')

train_results_car = model_car.train(
    data=car_only_yaml,
    epochs=40,
    imgsz=640,
    batch=16,
    lr0=0.001,
    freeze=0,
    workers=2,
    cache='ram',
    device=0,
    exist_ok=True,
    pretrained=True,
    cos_lr=True,
    name='train_car_only'
)

print('Car-only training complete.')
print(os.listdir('/content/kitti3k_car_only'))

Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/kitti3k_car_only/kitti3k_car_only.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/config3_best.pt, momentum=0.937, mosaic=1.0, multi_scale=0

In [ ]:
## CONFIG W/ TEST SET

# evaluate car-only model on test set
car_only_model = YOLO('/content/runs/detect/train_car_only/weights/best.pt')

test_metrics_car = car_only_model.val(
    data=car_only_yaml,
    split='test',
    imgsz=640,
    conf=0.001,
    iou=0.7,
    exist_ok=True,
    name='test_car_only'
)

print('Car-only model test set results:')
print(f'Car AP50:    {test_metrics_car.box.map50:.3f}')
print(f'Car AP50-95: {test_metrics_car.box.map:.3f}')

Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 124.2±20.7 MB/s, size: 827.6 KB)
val: Scanning /content/kitti3k_car_only/labels/test... 450 images, 56 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 450/450 209.8it/s 2.1s
val: New cache created: /content/kitti3k_car_only/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 3.6it/s 8.1s
                   all        450       1795       0.93      0.834      0.931      0.714
Speed: 1.1ms preprocess, 3.2ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to /content/runs/detect/test_car_only
Car-only model test set results:
Car AP50:    0.931
Car AP50-95: 0.714


In [ ]:
## COMPARISON TABLE

comparison_table = pd.DataFrame({
    'Model': ['Three-class Config 3 (test set)', 'One-class Car-only (test set)'],
    'Car AP50': [0.921, 0.931],
    'Car AP50-95': [0.692, 0.714],
})
comparison_table

,Model,Car AP50,Car AP50-95
0,Three-class Config 3 (test set),0.921,0.692
1,One-class Car-only (test set),0.931,0.714


In [ ]:
## SAVE

import shutil
shutil.copy('/content/runs/detect/train_car_only/weights/best.pt',
            '/content/drive/MyDrive/car_only_best.pt')
print('Car-only model saved.')

Car-only model saved.
